# Module 03 — Unsupervised & Statistical Learning
## 03-10: Bayesian Inference & Probabilistic Thinking

**Objective:** Implement Bayesian inference from scratch — prior specification, likelihood computation, posterior derivation, and predictive distributions — using conjugate prior pairs, and build full Bayesian linear regression with uncertainty quantification.

**Prerequisites:** 1-07 (Probability & Statistics), 3-06 (GMMs & EM — probabilistic reasoning)

## Part 0 — Setup & Prerequisites

This notebook owns **conjugate priors**, **posterior inference**, **MAP estimation**, **predictive distributions**, and **Bayesian linear regression**. MAP estimation was introduced in 1-07 with a single formula; here we derive it properly as the mode of the posterior. The EM algorithm (3-06) is the variational Bayes perspective on the same ideas — the E-step computes approximate posteriors and the M-step maximises expected log-likelihood. Bayesian thinking connects forward to variational autoencoders (Module 11) where the ELBO is a Bayesian evidence lower bound, and to reward modeling in RLHF (Module 13) where posterior uncertainty over preferences matters.

**Prerequisites:** 1-07 (Probability & Statistics), 3-06 (GMMs & EM Algorithm)

In [ ]:
import sys
import time
import warnings
import random
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats
from scipy.special import betaln
from sklearn.linear_model import Ridge, BayesianRidge
from sklearn.preprocessing import PolynomialFeatures
import torch

print(f'Python : {sys.version.split()[0]}')
print(f'NumPy  : {np.__version__}')
print(f'SciPy  : (imported)')
print(f'PyTorch: {torch.__version__}')

In [ ]:
import scipy
import sys
print(f'Python : {sys.version.split()[0]}')
print(f'NumPy  : {np.__version__}')
print(f'SciPy  : {scipy.__version__}')
print(f'PyTorch: {torch.__version__}')

In [ ]:
# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 1103
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ── Experiment constants ──────────────────────────────────────────────────────
# Beta-Binomial (coin-flip)
COIN_TRUE_P     = 0.65   # True probability of heads
N_COIN_FLIPS    = 100    # Total coin flip observations
# Normal-Normal (known variance)
TRUE_MU         = 3.0    # True population mean
KNOWN_SIGMA     = 1.5    # Known observation noise std
N_NORMAL_OBS    = 50     # Normal observations
# Bayesian linear regression
N_LINREG        = 40     # Training samples for regression
POLY_DEGREE     = 4      # Polynomial feature degree
NOISE_SIGMA     = 0.4    # Regression observation noise
ALPHA_PRIOR     = 2.0    # Weight prior precision (lambda = alpha/beta_noise)
BETA_NOISE      = 1.0 / (NOISE_SIGMA ** 2)  # Observation precision
# Grid resolution for density plots
N_GRID          = 400

print('Configuration loaded.')
print(f'  Coin true P      = {COIN_TRUE_P}')
print(f'  Normal true mu   = {TRUE_MU}')
print(f'  LinReg noise std = {NOISE_SIGMA}')

### Data Generation & Exploration

We generate three synthetic datasets — all controlled experiments where we know the ground truth parameters — to verify that our Bayesian estimates recover the true values.

In [ ]:
rng = np.random.RandomState(SEED)

# ── Dataset 1: Coin flips (Bernoulli with p = COIN_TRUE_P) ────────────────────
coin_flips: np.ndarray = rng.binomial(1, COIN_TRUE_P, size=N_COIN_FLIPS)
n_heads_total: int = int(coin_flips.sum())
n_tails_total: int = N_COIN_FLIPS - n_heads_total
print(f'Coin flips: {N_COIN_FLIPS} tosses | heads={n_heads_total} ({n_heads_total/N_COIN_FLIPS:.1%}) | true p={COIN_TRUE_P}')

# ── Dataset 2: Normal observations (mu = TRUE_MU, sigma = KNOWN_SIGMA) ────────
normal_obs: np.ndarray = rng.normal(TRUE_MU, KNOWN_SIGMA, size=N_NORMAL_OBS)
print(f'Normal obs: n={N_NORMAL_OBS} | sample_mean={normal_obs.mean():.3f} | true_mu={TRUE_MU}')

# ── Dataset 3: Noisy polynomial regression ────────────────────────────────────
X_linreg: np.ndarray = rng.uniform(-3, 3, size=N_LINREG)
y_linreg: np.ndarray = np.sin(X_linreg) + rng.normal(0, NOISE_SIGMA, size=N_LINREG)
X_test_linreg: np.ndarray = np.linspace(-3.5, 3.5, 200)
y_true_linreg: np.ndarray = np.sin(X_test_linreg)
print(f'LinReg: n_train={N_LINREG} | y range=[{y_linreg.min():.2f}, {y_linreg.max():.2f}]')

# ── Quick EDA plot ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

ax = axes[0]
cumulative_heads = np.cumsum(coin_flips) / (np.arange(N_COIN_FLIPS) + 1)
ax.plot(np.arange(1, N_COIN_FLIPS + 1), cumulative_heads, 'b-')
ax.axhline(COIN_TRUE_P, color='r', ls='--', label=f'True p={COIN_TRUE_P}')
ax.set_xlabel('Number of flips')
ax.set_ylabel('Cumulative proportion heads')
ax.set_title('Coin Flip Data')
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.hist(normal_obs, bins=15, color='steelblue', alpha=0.7, density=True)
xs = np.linspace(normal_obs.min() - 1, normal_obs.max() + 1, 200)
ax.plot(xs, stats.norm.pdf(xs, TRUE_MU, KNOWN_SIGMA), 'r-', lw=2, label=f'True N({TRUE_MU},{KNOWN_SIGMA})')
ax.axvline(normal_obs.mean(), color='g', ls='--', label=f'Sample mean={normal_obs.mean():.2f}')
ax.set_xlabel('Observation')
ax.set_ylabel('Density')
ax.set_title('Normal Observations')
ax.legend(fontsize=8)

ax = axes[2]
ax.scatter(X_linreg, y_linreg, s=25, alpha=0.7, c='steelblue', label='Training data')
ax.plot(X_test_linreg, y_true_linreg, 'r-', lw=2, label='True sin(x)')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('Noisy Regression Data')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('bayesian_data.png', dpi=100, bbox_inches='tight')
plt.show()

---
## Part 1 — Bayesian Inference from Scratch

### 1.1 Bayesian Recipe

The fundamental equation of Bayesian inference is Bayes' theorem:

$$\underbrace{p(\theta \mid \mathcal{D})}_{\text{posterior}} = \frac{\overbrace{p(\mathcal{D} \mid \theta)}^{\text{likelihood}} \; \overbrace{p(\theta)}^{\text{prior}}}{\underbrace{p(\mathcal{D})}_{\text{evidence}}}$$

**Conjugate priors** are special priors where the posterior has the same functional form as the prior — they make posterior computation analytic (no sampling required).

### 1.2 Beta-Binomial Conjugate Pair

**Model:** $X \sim \text{Binomial}(n, \theta)$, $\theta \sim \text{Beta}(\alpha, \beta)$.

**Posterior:** After observing $k$ heads in $n$ trials, the posterior is:
$$\theta \mid \mathcal{D} \sim \text{Beta}(\alpha + k, \; \beta + n - k)$$

This closed-form update is the key advantage of conjugate priors — no integrals needed.

**MAP estimate:** The mode of $\text{Beta}(\alpha, \beta)$ is $\hat{\theta}_{\text{MAP}} = \frac{\alpha - 1}{\alpha + \beta - 2}$ (only valid for $\alpha, \beta > 1$).

**MLE:** $\hat{\theta}_{\text{MLE}} = k/n$ — ignores the prior entirely.

**Posterior mean:** $\mathbb{E}[\theta \mid \mathcal{D}] = \frac{\alpha + k}{\alpha + \beta + n}$ — a weighted average of prior mean and MLE.

### 1.3 Normal-Normal Conjugate Pair (Known Variance)

**Model:** $x_i \sim \mathcal{N}(\mu, \sigma^2)$ with known $\sigma^2$; prior $\mu \sim \mathcal{N}(\mu_0, \tau_0^2)$.

**Posterior:** After observing $n$ samples with mean $\bar{x}$:
$$\mu \mid \mathcal{D} \sim \mathcal{N}\!\left(\mu_n, \tau_n^2\right), \quad \tau_n^{-2} = \tau_0^{-2} + n/\sigma^2, \quad \mu_n = \tau_n^2 \left( \frac{\mu_0}{\tau_0^2} + \frac{n \bar{x}}{\sigma^2}\right)$$

## Part 1 — Bayesian Inference from Scratch

We start with the two most important conjugate pairs in Bayesian statistics: **Beta-Binomial** (for coin-flip/proportion problems) and **Normal-Normal** (for estimating Gaussian means). These are the building blocks for understanding how priors and likelihoods combine into posteriors.

In [ ]:
# ── Beta-Binomial primitives ──────────────────────────────────────────────────

def beta_log_prob(theta: np.ndarray, alpha: float, beta: float) -> np.ndarray:
    '''Log probability density of Beta(alpha, beta) at theta.

    Uses the log Beta function for numerical stability:
        log p(theta) = (alpha-1) log(theta) + (beta-1) log(1-theta) - log B(alpha, beta)

    Args:
        theta: Values in (0, 1), shape (n,) or scalar.
        alpha: First shape parameter (> 0).
        beta: Second shape parameter (> 0).

    Returns:
        Log density values, same shape as theta.
    '''
    theta = np.clip(theta, 1e-10, 1 - 1e-10)
    log_norm: float = betaln(alpha, beta)
    return (alpha - 1) * np.log(theta) + (beta - 1) * np.log(1 - theta) - log_norm


def binomial_log_likelihood(theta: np.ndarray, n_heads: int, n_trials: int) -> np.ndarray:
    '''Log likelihood of Binomial(n_trials, theta) model at theta.

    Proportional to: k * log(theta) + (n-k) * log(1-theta).
    Constant binomial coefficient dropped since it does not depend on theta.

    Args:
        theta: Probability grid, shape (n,).
        n_heads: Number of observed heads k.
        n_trials: Total trials n.

    Returns:
        Log likelihood values (unnormalised), same shape as theta.
    '''
    theta = np.clip(theta, 1e-10, 1 - 1e-10)
    return n_heads * np.log(theta) + (n_trials - n_heads) * np.log(1 - theta)


def beta_posterior(alpha_prior: float, beta_prior: float,
                   n_heads: int, n_tails: int) -> tuple:
    '''Compute Beta-Binomial posterior parameters.

    Posterior is Beta(alpha + n_heads, beta + n_tails).

    Args:
        alpha_prior: Prior alpha (pseudo-counts for heads).
        beta_prior: Prior beta (pseudo-counts for tails).
        n_heads: Observed heads.
        n_tails: Observed tails.

    Returns:
        Tuple (alpha_post, beta_post).
    '''
    return alpha_prior + n_heads, beta_prior + n_tails


def beta_statistics(alpha: float, beta: float) -> dict:
    '''Compute mean, mode, variance and 95% credible interval of Beta(alpha, beta).

    Args:
        alpha: Beta shape parameter 1.
        beta: Beta shape parameter 2.

    Returns:
        Dict with keys 'mean', 'mode', 'variance', 'ci_lo', 'ci_hi'.
    '''
    mean: float = alpha / (alpha + beta)
    # Mode only defined when alpha, beta > 1
    if alpha > 1 and beta > 1:
        mode: float = (alpha - 1) / (alpha + beta - 2)
    else:
        mode = float('nan')
    variance: float = alpha * beta / ((alpha + beta) ** 2 * (alpha + beta + 1))
    ci_lo: float = float(stats.beta.ppf(0.025, alpha, beta))
    ci_hi: float = float(stats.beta.ppf(0.975, alpha, beta))
    return {'mean': mean, 'mode': mode, 'variance': variance, 'ci_lo': ci_lo, 'ci_hi': ci_hi}


def beta_posterior_predictive(alpha: float, beta: float) -> float:
    '''Posterior predictive probability of heads for next flip.

    For Beta-Binomial, the predictive probability of one more head equals
    the posterior mean: E[theta | D] = alpha / (alpha + beta).

    Args:
        alpha: Posterior alpha parameter.
        beta: Posterior beta parameter.

    Returns:
        Probability of heads for the next trial.
    '''
    return alpha / (alpha + beta)


# ── Verify on toy example ─────────────────────────────────────────────────────
alpha_p, beta_p = 2.0, 2.0  # Uniform-ish prior
k_obs, n_obs = 7, 10         # Observed 7 heads in 10 flips
alpha_post, beta_post = beta_posterior(alpha_p, beta_p, k_obs, n_obs - k_obs)
stats_post = beta_statistics(alpha_post, beta_post)
mle = k_obs / n_obs

print('Beta-Binomial example (prior=Beta(2,2), data: 7H/10):')
print(f'  Prior         : Beta({alpha_p}, {beta_p}) | mean={alpha_p/(alpha_p+beta_p):.3f}')
print(f'  Posterior     : Beta({alpha_post:.1f}, {beta_post:.1f})')
print(f'  MLE (k/n)     : {mle:.3f}')
print(f'  Posterior mean: {stats_post["mean"]:.3f}')
print(f'  Posterior mode (MAP): {stats_post["mode"]:.3f}')
print(f'  95% CI        : [{stats_post["ci_lo"]:.3f}, {stats_post["ci_hi"]:.3f}]')

In [ ]:
# ── Normal-Normal conjugate pair (known variance) ─────────────────────────────

def normal_posterior_known_var(
    mu_0: float,
    tau_0_sq: float,
    sigma_sq: float,
    data: np.ndarray,
) -> tuple:
    '''Compute Normal-Normal posterior for mu with known observation variance.

    Prior: mu ~ N(mu_0, tau_0^2)
    Likelihood: x_i ~ N(mu, sigma^2) i.i.d.
    Posterior: mu | D ~ N(mu_n, tau_n^2)

    Posterior precision = prior precision + data precision:
        tau_n^{-2} = tau_0^{-2} + n / sigma^2
        mu_n = tau_n^2 * (mu_0 / tau_0^2 + n * xbar / sigma^2)

    Args:
        mu_0: Prior mean.
        tau_0_sq: Prior variance.
        sigma_sq: Known observation variance.
        data: Observed data array of shape (n,).

    Returns:
        Tuple (mu_n, tau_n_sq): posterior mean and posterior variance.
    '''
    n: int = len(data)
    x_bar: float = float(data.mean()) if n > 0 else mu_0
    precision_prior: float = 1.0 / tau_0_sq
    precision_data: float = n / sigma_sq
    tau_n_sq: float = 1.0 / (precision_prior + precision_data)
    mu_n: float = tau_n_sq * (mu_0 * precision_prior + x_bar * precision_data)
    return mu_n, tau_n_sq


def normal_log_prob(x: np.ndarray, mu: float, sigma_sq: float) -> np.ndarray:
    '''Log density of N(mu, sigma_sq) evaluated at x.

    Args:
        x: Evaluation points, shape (n,).
        mu: Mean.
        sigma_sq: Variance.

    Returns:
        Log density values, same shape as x.
    '''
    return -0.5 * np.log(2 * np.pi * sigma_sq) - (x - mu) ** 2 / (2 * sigma_sq)


def normal_posterior_predictive(mu_n: float, tau_n_sq: float, sigma_sq: float) -> tuple:
    '''Posterior predictive distribution for a new observation x_new.

    Integrating out mu: p(x_new | D) = N(x_new | mu_n, tau_n^2 + sigma^2).

    Args:
        mu_n: Posterior mean of mu.
        tau_n_sq: Posterior variance of mu.
        sigma_sq: Known observation variance.

    Returns:
        Tuple (pred_mean, pred_variance) of the posterior predictive.
    '''
    return mu_n, tau_n_sq + sigma_sq


# ── Verify Normal-Normal on toy data ─────────────────────────────────────────
mu_0_toy, tau_0_sq_toy = 0.0, 4.0   # Prior: N(0, 4)
sigma_sq_toy = KNOWN_SIGMA ** 2
data_toy: np.ndarray = np.array([3.2, 2.8, 3.5, 2.9, 3.1])  # 5 observations

mu_n_toy, tau_n_sq_toy = normal_posterior_known_var(
    mu_0_toy, tau_0_sq_toy, sigma_sq_toy, data_toy
)
pred_mean_toy, pred_var_toy = normal_posterior_predictive(mu_n_toy, tau_n_sq_toy, sigma_sq_toy)

print('Normal-Normal example (prior=N(0,4), sigma=1.5, n=5 observations):')
print(f'  Prior mean / std    : {mu_0_toy:.2f} / {np.sqrt(tau_0_sq_toy):.2f}')
print(f'  Sample mean         : {data_toy.mean():.3f}')
print(f'  Posterior mean      : {mu_n_toy:.3f}')
print(f'  Posterior std       : {np.sqrt(tau_n_sq_toy):.3f}')
print(f'  Predictive mean/std : {pred_mean_toy:.3f} / {np.sqrt(pred_var_toy):.3f}')

In [ ]:
# ── Sequential Normal-Normal updating: posterior narrowing ────────────────────
print('=== Sequential Normal-Normal Updating ===\n')

mu_0_seq: float = 0.0
tau_0_sq_seq: float = 9.0    # Wide prior N(0, 9)
sigma_sq_seq: float = KNOWN_SIGMA ** 2

snap_ns_nn: list = [0, 1, 2, 5, 10, 20, 50]
mu_grid_nn: np.ndarray = np.linspace(TRUE_MU - 4, TRUE_MU + 4, N_GRID)
bnn_seq = BayesianNormalNormal(mu_0_seq, tau_0_sq_seq, sigma_sq_seq)
snapshots_nn: list = [(0, mu_0_seq, tau_0_sq_seq)]
prev_nn: int = 0

for sn in snap_ns_nn[1:]:
    chunk_nn = normal_obs[prev_nn:sn]
    bnn_seq.update(chunk_nn)
    prev_nn = sn
    snapshots_nn.append((sn, bnn_seq.mu_n_, bnn_seq.tau_n_sq_))
    ci_lo_nn, ci_hi_nn = bnn_seq.credible_interval(0.95)
    print(f'  n={sn:3d} | post_mean={bnn_seq.mu_n_:.3f} | post_std={bnn_seq.posterior_std():.3f} | '
          f'95%CI=[{ci_lo_nn:.3f},{ci_hi_nn:.3f}]')

print(f'\nTrue mu = {TRUE_MU} | MLE (sample mean) = {normal_obs.mean():.3f}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
colors_nn = plt.cm.cool(np.linspace(0, 1, len(snapshots_nn)))
for idx_nn, (sn, mu_n, tau_n_sq) in enumerate(snapshots_nn):
    pdf_nn = np.exp(normal_log_prob(mu_grid_nn, mu_n, tau_n_sq))
    lw_nn = 2.5 if idx_nn == 0 or sn == snap_ns_nn[-1] else 1.5
    ls_nn = '--' if idx_nn == 0 else '-'
    ax.plot(mu_grid_nn, pdf_nn, color=colors_nn[idx_nn], lw=lw_nn, ls=ls_nn,
            label=f'n={sn}' if sn <= 20 else f'n={sn} (final)')
ax.axvline(TRUE_MU, color='r', ls=':', lw=2, label=f'True mu={TRUE_MU}')
ax.axvline(normal_obs.mean(), color='gray', ls='--', lw=1.5, label=f'MLE={normal_obs.mean():.2f}')
ax.set_xlabel('mu')
ax.set_ylabel('Density')
ax.set_title('Normal-Normal Posterior Narrowing')
ax.legend(fontsize=8, ncol=2)
ax.grid(True, alpha=0.3)

ax = axes[1]
sn_vals = [s[0] for s in snapshots_nn]
mu_vals = [s[1] for s in snapshots_nn]
std_vals = [np.sqrt(s[2]) for s in snapshots_nn]
ax.plot(sn_vals, mu_vals, 'b-o', ms=5, label='Posterior mean')
ax.fill_between(sn_vals,
                [m - 2 * s for m, s in zip(mu_vals, std_vals)],
                [m + 2 * s for m, s in zip(mu_vals, std_vals)],
                alpha=0.25, color='blue', label='+/-2 post std')
ax.axhline(TRUE_MU, color='r', ls='--', lw=1.5, label=f'True mu={TRUE_MU}')
ax.axhline(normal_obs.mean(), color='gray', ls=':', lw=1.5, label='MLE')
ax.set_xlabel('Number of observations n')
ax.set_ylabel('Estimated mu')
ax.set_title('Posterior Convergence to True Value')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('normal_sequential_update.png', dpi=100, bbox_inches='tight')
plt.show()

print('\nPosterior std decreases as O(1/sqrt(n)), reflecting increasing certainty.')
print('With more data, the posterior mean converges to the MLE (data dominates prior).')

### Sequential Bayesian Updating

A key strength of Bayesian inference is **online updating**: each new observation updates the posterior, which becomes the prior for the next observation. The order of observations does not matter — the final posterior is identical whether we update sequentially or in batch.

In [ ]:
# ── Sequential Bayesian updating: Beta-Binomial ───────────────────────────────
print('=== Sequential Bayesian Updating: Beta-Binomial ===\n')

alpha_seq, beta_seq = 1.0, 1.0   # Uniform prior (no information)
theta_grid: np.ndarray = np.linspace(0.01, 0.99, N_GRID)

snapshot_ns: list = [0, 1, 3, 7, 15, 30, 60, 100]
snapshots: list = []   # (n, alpha, beta, stats_dict)

alpha_curr, beta_curr = alpha_seq, beta_seq
prev_n: int = 0
for snap_n in snapshot_ns:
    if snap_n > prev_n:
        chunk = coin_flips[prev_n:snap_n]
        n_h = int(chunk.sum())
        n_t = len(chunk) - n_h
        alpha_curr, beta_curr = beta_posterior(alpha_curr, beta_curr, n_h, n_t)
        prev_n = snap_n
    s = beta_statistics(alpha_curr, beta_curr)
    snapshots.append((snap_n, alpha_curr, beta_curr, s))
    mode_str = f'{s["mode"]:.3f}' if not np.isnan(s['mode']) else 'n/a'
    print(f'  n={snap_n:3d} | Beta({alpha_curr:.1f},{beta_curr:.1f}) | '
          f'mean={s["mean"]:.3f} | mode={mode_str} | '
          f'95%CI=[{s["ci_lo"]:.3f},{s["ci_hi"]:.3f}]')

# Plot posterior evolution
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for idx, (snap_n, a, b, st) in enumerate(snapshots):
    ax = axes[idx // 4, idx % 4]
    log_prior = beta_log_prob(theta_grid, alpha_seq, beta_seq)
    log_post = beta_log_prob(theta_grid, a, b)
    ax.plot(theta_grid, np.exp(log_prior), 'g--', lw=1.5, alpha=0.6, label='Prior')
    ax.plot(theta_grid, np.exp(log_post), 'b-', lw=2, label='Posterior')
    ax.axvline(COIN_TRUE_P, color='r', ls=':', lw=2, label='True p')
    ax.axvline(st['mean'], color='orange', ls='--', lw=1.5, label='Post mean')
    ax.set_xlabel('theta')
    ax.set_title(f'n={snap_n}: Beta({a:.0f},{b:.0f})', fontsize=9)
    if idx == 0:
        ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

plt.suptitle('Sequential Bayesian Updating — Beta-Binomial (coin flip)', fontsize=12)
plt.tight_layout()
plt.savefig('beta_sequential_update.png', dpi=100, bbox_inches='tight')
plt.show()

print(f'\nFinal posterior: Beta({alpha_curr:.1f},{beta_curr:.1f})')
print(f'True p={COIN_TRUE_P} | Posterior mean={alpha_curr/(alpha_curr+beta_curr):.3f}')

### MAP Estimation — Posterior Mode vs Full Posterior

**Maximum A Posteriori (MAP)** estimation finds the single parameter value that maximises the posterior, discarding all information about posterior shape. It is equivalent to regularised maximum likelihood:
- Beta-Binomial MAP corresponds to adding $\alpha - 1$ pseudo-head-counts and $\beta - 1$ pseudo-tail-counts.
- Normal-Normal MAP equals $L_2$-penalised mean estimation (ridge regression for linear models).

MAP is a point estimate; the full posterior captures uncertainty. As $n \to \infty$, MLE, MAP, and the posterior mean all converge.

In [ ]:
# ── MLE vs MAP vs Posterior Mean comparison ───────────────────────────────────
print('=== MLE vs MAP vs Posterior Mean: Beta-Binomial ===\n')

# Sweep over increasing dataset sizes
prior_scenarios: list = [
    ('Weak prior', 1.1, 1.1),
    ('Moderate prior', 3.0, 3.0),
    ('Strong prior', 10.0, 10.0),
]
ns_sweep: np.ndarray = np.array([1, 3, 5, 10, 20, 40, 70, 100])

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, (scenario_name, a_prior, b_prior) in zip(axes, prior_scenarios):
    mle_vals: list = []
    map_vals: list = []
    mean_vals: list = []
    for n_sweep in ns_sweep:
        flips_sub = coin_flips[:n_sweep]
        k_sub = int(flips_sub.sum())
        a_post_s, b_post_s = beta_posterior(a_prior, b_prior, k_sub, n_sweep - k_sub)
        st_s = beta_statistics(a_post_s, b_post_s)
        mle_vals.append(k_sub / n_sweep)
        map_vals.append(st_s['mode'] if not np.isnan(st_s['mode']) else float('nan'))
        mean_vals.append(st_s['mean'])
    ax.plot(ns_sweep, mle_vals, 'b-o', ms=4, label='MLE')
    ax.plot(ns_sweep, map_vals, 'r-s', ms=4, label='MAP')
    ax.plot(ns_sweep, mean_vals, 'g-^', ms=4, label='Post mean')
    ax.axhline(COIN_TRUE_P, color='gray', ls='--', lw=1.5, label=f'True p={COIN_TRUE_P}')
    ax.set_xlabel('n (samples observed)')
    ax.set_ylabel('Estimate of p')
    ax.set_title(f'{scenario_name}\nBeta({a_prior},{b_prior})')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0.0, 1.0)

plt.suptitle('MLE vs MAP vs Posterior Mean: Convergence with Data', fontsize=12)
plt.tight_layout()
plt.savefig('mle_map_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

print('Key insight: MAP shrinks toward the prior; with more data all estimators converge.')

### 1.4 Bayesian Linear Regression

For a linear model $\mathbf{y} = \Phi \mathbf{w} + \boldsymbol{\varepsilon}$, $\boldsymbol{\varepsilon} \sim \mathcal{N}(0, \beta^{-1}\mathbf{I})$ with Gaussian prior $\mathbf{w} \sim \mathcal{N}(\mathbf{0}, \alpha^{-1}\mathbf{I})$, the posterior is also Gaussian:

$$\mathbf{w} \mid \mathcal{D} \sim \mathcal{N}(\mathbf{m}_N, \mathbf{S}_N)$$

$$\mathbf{S}_N = \left(\alpha \mathbf{I} + \beta \Phi^\top \Phi\right)^{-1}, \quad \mathbf{m}_N = \beta \mathbf{S}_N \Phi^\top \mathbf{y}$$

The **posterior predictive** for a new input $\phi^*$ is:
$$p(y^* \mid \phi^*, \mathcal{D}) = \mathcal{N}\!\left(\mathbf{m}_N^\top \phi^*, \; \beta^{-1} + (\phi^*)^\top \mathbf{S}_N \phi^*\right)$$

The variance has two terms: **aleatoric** ($\beta^{-1}$, irreducible noise) and **epistemic** ($(\phi^*)^\top \mathbf{S}_N \phi^*$, uncertainty about $\mathbf{w}$).

In [ ]:
# ── Bayesian Linear Regression primitives ─────────────────────────────────────

def build_polynomial_features(X: np.ndarray, degree: int) -> np.ndarray:
    '''Build polynomial design matrix [1, x, x^2, ..., x^degree].

    Args:
        X: Input array of shape (n,) or (n, 1).
        degree: Polynomial degree.

    Returns:
        Design matrix of shape (n, degree + 1).
    '''
    X_flat: np.ndarray = X.ravel()
    return np.column_stack([X_flat ** d for d in range(degree + 1)])


def bayesian_linreg_posterior(
    Phi: np.ndarray,
    y: np.ndarray,
    alpha: float,
    beta_noise: float,
) -> tuple:
    '''Compute exact Gaussian posterior over weights w for Bayesian linear regression.

    Prior: w ~ N(0, alpha^{-1} I)
    Likelihood: y | Phi, w ~ N(Phi w, beta_noise^{-1} I)
    Posterior: w | D ~ N(m_N, S_N)

    S_N = (alpha I + beta_noise Phi^T Phi)^{-1}
    m_N = beta_noise S_N Phi^T y

    Args:
        Phi: Design matrix (n_samples, n_features).
        y: Target vector (n_samples,).
        alpha: Weight prior precision (inverse variance).
        beta_noise: Observation precision (1 / noise_variance).

    Returns:
        Tuple (m_N, S_N): posterior mean (n_features,) and covariance (n_features, n_features).
    '''
    n_features: int = Phi.shape[1]
    S_N_inv: np.ndarray = alpha * np.eye(n_features) + beta_noise * Phi.T @ Phi
    S_N: np.ndarray = np.linalg.inv(S_N_inv)
    m_N: np.ndarray = beta_noise * S_N @ Phi.T @ y
    return m_N, S_N


def bayesian_linreg_predict(
    Phi_test: np.ndarray,
    m_N: np.ndarray,
    S_N: np.ndarray,
    beta_noise: float,
) -> tuple:
    '''Posterior predictive mean and variance for test inputs.

    p(y* | phi*, D) = N(y* | m_N^T phi*, sigma^2_*(phi*))
    sigma^2_*(phi*) = 1/beta_noise + phi*^T S_N phi*

    Args:
        Phi_test: Test design matrix (n_test, n_features).
        m_N: Posterior mean weights (n_features,).
        S_N: Posterior weight covariance (n_features, n_features).
        beta_noise: Observation precision.

    Returns:
        Tuple (pred_mean, pred_std): predictive mean and std, each shape (n_test,).
    '''
    pred_mean: np.ndarray = Phi_test @ m_N                           # (n_test,)
    aleatoric: float = 1.0 / beta_noise                              # observation noise
    epistemic: np.ndarray = np.sum(Phi_test @ S_N * Phi_test, axis=1)  # (n_test,)
    pred_var: np.ndarray = aleatoric + epistemic
    return pred_mean, np.sqrt(np.maximum(pred_var, 0.0))


def bayesian_linreg_map(Phi: np.ndarray, y: np.ndarray,
                         alpha: float, beta_noise: float) -> np.ndarray:
    '''MAP estimate of weights: equivalent to ridge regression.

    w_MAP = (alpha/beta_noise I + Phi^T Phi)^{-1} Phi^T y
           = argmin { ||y - Phi w||^2 + (alpha/beta_noise) ||w||^2 }

    Args:
        Phi: Design matrix (n_samples, n_features).
        y: Target vector (n_samples,).
        alpha: Weight prior precision.
        beta_noise: Observation precision.

    Returns:
        MAP weight vector (n_features,).
    '''
    lambda_reg: float = alpha / beta_noise
    n_features: int = Phi.shape[1]
    A: np.ndarray = Phi.T @ Phi + lambda_reg * np.eye(n_features)
    return np.linalg.solve(A, Phi.T @ y)


# ── Quick verification ─────────────────────────────────────────────────────────
Phi_toy = build_polynomial_features(X_linreg[:10], degree=2)  # (10, 3)
y_toy = y_linreg[:10]
m_toy, S_toy = bayesian_linreg_posterior(Phi_toy, y_toy, ALPHA_PRIOR, BETA_NOISE)
w_map_toy = bayesian_linreg_map(Phi_toy, y_toy, ALPHA_PRIOR, BETA_NOISE)

print('Bayesian LinReg verification (n=10, degree=2):')
print(f'  Posterior mean (m_N): {m_toy.round(4)}')
print(f'  MAP estimate (w_MAP): {w_map_toy.round(4)}')
print(f'  |m_N - w_MAP|       : {np.abs(m_toy - w_map_toy).max():.2e}  (should be ~0)')
print(f'  S_N shape           : {S_toy.shape}')

---
## Part 2 — Assembled Bayesian Classes

We wrap the primitives into three reusable classes with sklearn-compatible interfaces.

## Part 2 — Putting It All Together

We now wrap our conjugate update rules into reusable classes — `BayesianBetaBinomial`, `BayesianNormalNormal`, and `BayesianLinearRegression` — each with `update()` and `predict()` methods.

In [ ]:
class BayesianBetaBinomial:
    '''Online Bayesian inference for Bernoulli probability with Beta-Binomial model.

    Maintains a Beta posterior over p = P(heads), updated exactly via
    the conjugate update rule after each batch of observations.

    Args:
        alpha_prior: Initial prior alpha (pseudo-head count). Default: 1 (uniform).
        beta_prior: Initial prior beta (pseudo-tail count). Default: 1 (uniform).

    Attributes:
        alpha_: Current posterior alpha.
        beta_: Current posterior beta.
        n_observed_: Total number of trials seen.
    '''

    def __init__(self, alpha_prior: float = 1.0, beta_prior: float = 1.0) -> None:
        '''Initialise BayesianBetaBinomial.

        Args:
            alpha_prior: Initial prior alpha.
            beta_prior: Initial prior beta.
        '''
        self.alpha_prior = alpha_prior
        self.beta_prior = beta_prior
        self.alpha_: float = alpha_prior
        self.beta_: float = beta_prior
        self.n_observed_: int = 0

    def update(self, observations: np.ndarray) -> 'BayesianBetaBinomial':
        '''Update posterior with new binary observations.

        Args:
            observations: Binary array (0=tails, 1=heads).

        Returns:
            Self (for chaining).
        '''
        n_heads: int = int(np.sum(observations))
        n_tails: int = len(observations) - n_heads
        self.alpha_, self.beta_ = beta_posterior(self.alpha_, self.beta_, n_heads, n_tails)
        self.n_observed_ += len(observations)
        return self

    def posterior_mean(self) -> float:
        '''Posterior mean of p.

        Returns:
            Float in (0, 1).
        '''
        return float(self.alpha_ / (self.alpha_ + self.beta_))

    def posterior_map(self) -> float:
        '''Posterior mode (MAP estimate) of p.

        Returns:
            Float in (0, 1), or nan if alpha <= 1 or beta <= 1.
        '''
        s = beta_statistics(self.alpha_, self.beta_)
        return float(s['mode'])

    def credible_interval(self, coverage: float = 0.95) -> tuple:
        '''Equal-tailed credible interval for p.

        Args:
            coverage: Nominal coverage probability (default 0.95).

        Returns:
            Tuple (lo, hi).
        '''
        alpha_ci: float = (1 - coverage) / 2
        lo = float(stats.beta.ppf(alpha_ci, self.alpha_, self.beta_))
        hi = float(stats.beta.ppf(1 - alpha_ci, self.alpha_, self.beta_))
        return lo, hi

    def posterior_pdf(self, theta_grid: np.ndarray) -> np.ndarray:
        '''Posterior density on a grid of theta values.

        Args:
            theta_grid: Array of values in (0, 1).

        Returns:
            Density values, same shape as theta_grid.
        '''
        return np.exp(beta_log_prob(theta_grid, self.alpha_, self.beta_))

    def predictive_p_heads(self) -> float:
        '''Predictive probability of heads for next trial.

        Returns:
            Float in (0, 1), equals posterior mean.
        '''
        return self.posterior_mean()

In [ ]:
class BayesianNormalNormal:
    '''Online Bayesian inference for Normal mean with known variance.

    Args:
        mu_0: Prior mean.
        tau_0_sq: Prior variance.
        sigma_sq: Known observation variance.

    Attributes:
        mu_n_: Current posterior mean.
        tau_n_sq_: Current posterior variance.
        n_observed_: Total observations processed.
    '''

    def __init__(self, mu_0: float, tau_0_sq: float, sigma_sq: float) -> None:
        '''Initialise BayesianNormalNormal.

        Args:
            mu_0: Prior mean.
            tau_0_sq: Prior variance.
            sigma_sq: Known observation variance.
        '''
        self.mu_0 = mu_0
        self.tau_0_sq = tau_0_sq
        self.sigma_sq = sigma_sq
        self.mu_n_: float = mu_0
        self.tau_n_sq_: float = tau_0_sq
        self.n_observed_: int = 0

    def update(self, data: np.ndarray) -> 'BayesianNormalNormal':
        '''Update posterior with new continuous observations.

        Args:
            data: Observed data array.

        Returns:
            Self (for chaining).
        '''
        self.mu_n_, self.tau_n_sq_ = normal_posterior_known_var(
            self.mu_n_, self.tau_n_sq_, self.sigma_sq, data,
        )
        self.n_observed_ += len(data)
        return self

    def posterior_mean(self) -> float:
        '''Posterior mean of mu (also the MAP estimate for symmetric distributions).

        Returns:
            Current posterior mean.
        '''
        return self.mu_n_

    def posterior_std(self) -> float:
        '''Posterior standard deviation of mu.

        Returns:
            Square root of posterior variance.
        '''
        return float(np.sqrt(self.tau_n_sq_))

    def credible_interval(self, coverage: float = 0.95) -> tuple:
        '''Equal-tailed credible interval for mu.

        Args:
            coverage: Nominal coverage probability.

        Returns:
            Tuple (lo, hi).
        '''
        z: float = float(stats.norm.ppf(0.5 + coverage / 2))
        std_n = self.posterior_std()
        return self.mu_n_ - z * std_n, self.mu_n_ + z * std_n

    def posterior_pdf(self, mu_grid: np.ndarray) -> np.ndarray:
        '''Posterior density on a grid of mu values.

        Args:
            mu_grid: Evaluation points.

        Returns:
            Density values.
        '''
        return np.exp(normal_log_prob(mu_grid, self.mu_n_, self.tau_n_sq_))

In [ ]:
class BayesianLinearRegression:
    '''Bayesian linear regression with Gaussian prior over weights.

    Model:
        w ~ N(0, alpha^{-1} I)  (isotropic Gaussian prior)
        y | x, w ~ N(phi(x)^T w, beta_noise^{-1})  (Gaussian likelihood)

    Posterior and predictive distributions are computed analytically.

    Args:
        degree: Polynomial feature degree for the design matrix.
        alpha: Prior precision on weights (higher = stronger regularisation).
        beta_noise: Observation precision (1 / noise_variance).

    Attributes:
        m_N_: Posterior mean weights, shape (degree+1,).
        S_N_: Posterior weight covariance, shape (degree+1, degree+1).
    '''

    def __init__(
        self,
        degree: int = POLY_DEGREE,
        alpha: float = ALPHA_PRIOR,
        beta_noise: float = BETA_NOISE,
    ) -> None:
        '''Initialise BayesianLinearRegression.

        Args:
            degree: Polynomial degree for feature expansion.
            alpha: Prior precision over weights.
            beta_noise: Observation precision (1 / noise variance).
        '''
        self.degree = degree
        self.alpha = alpha
        self.beta_noise = beta_noise
        self.m_N_: np.ndarray | None = None
        self.S_N_: np.ndarray | None = None

    def fit(self, X: np.ndarray, y: np.ndarray) -> 'BayesianLinearRegression':
        '''Compute posterior over weights given training data.

        Args:
            X: Training inputs (n_samples,) or (n_samples, 1).
            y: Training targets (n_samples,).

        Returns:
            Self (for chaining).
        '''
        Phi: np.ndarray = build_polynomial_features(X, self.degree)
        self.m_N_, self.S_N_ = bayesian_linreg_posterior(Phi, y, self.alpha, self.beta_noise)
        return self

    def predict(self, X_test: np.ndarray) -> tuple:
        '''Predictive mean and standard deviation for test inputs.

        Args:
            X_test: Test inputs (n_test,) or (n_test, 1).

        Returns:
            Tuple (pred_mean, pred_std), each shape (n_test,).
        '''
        if self.m_N_ is None or self.S_N_ is None:
            raise RuntimeError('Call fit() before predict().')
        Phi_test: np.ndarray = build_polynomial_features(X_test, self.degree)
        return bayesian_linreg_predict(Phi_test, self.m_N_, self.S_N_, self.beta_noise)

    def map_weights(self, X: np.ndarray, y: np.ndarray) -> np.ndarray:
        '''Compute MAP weight estimate (equivalent to ridge regression).

        Args:
            X: Training inputs.
            y: Training targets.

        Returns:
            MAP weights, shape (degree+1,).
        '''
        Phi: np.ndarray = build_polynomial_features(X, self.degree)
        return bayesian_linreg_map(Phi, y, self.alpha, self.beta_noise)

    def log_marginal_likelihood(self, X: np.ndarray, y: np.ndarray) -> float:
        '''Log marginal likelihood (Bayesian model evidence).

        log p(y | X) = log N(y | 0, C)  where C = beta_noise^{-1} I + alpha^{-1} Phi Phi^T

        Args:
            X: Inputs (n_samples,).
            y: Targets (n_samples,).

        Returns:
            Scalar log marginal likelihood.
        '''
        Phi = build_polynomial_features(X, self.degree)
        n = len(y)
        C: np.ndarray = (1.0 / self.beta_noise) * np.eye(n) + (1.0 / self.alpha) * Phi @ Phi.T
        sign, log_det = np.linalg.slogdet(C)
        C_inv = np.linalg.inv(C)
        log_lml: float = -0.5 * (n * np.log(2 * np.pi) + log_det + float(y @ C_inv @ y))
        return log_lml


# ── Sanity check ──────────────────────────────────────────────────────────────
blr_san = BayesianLinearRegression(degree=2, alpha=ALPHA_PRIOR, beta_noise=BETA_NOISE)
blr_san.fit(X_linreg[:15], y_linreg[:15])
pred_m_san, pred_s_san = blr_san.predict(np.array([0.0, 1.0, -1.0]))

print('BayesianLinearRegression sanity check (n=15, degree=2):')
print(f'  Posterior mean weights: {blr_san.m_N_.round(4)}')
print(f'  Predictive mean at [0,1,-1]: {pred_m_san.round(4)}')
print(f'  Predictive std  at [0,1,-1]: {pred_s_san.round(4)}')

---
## Part 3 — Application

In [ ]:
# ── Beta-Binomial on full coin flip data ──────────────────────────────────────
print('=== Beta-Binomial on Full Coin Flip Dataset ===\n')

# Compare three priors: uniform, weakly informative, informative (wrong)
priors_bb: list = [
    ('Uniform (no info)', 1.0, 1.0),
    ('Weakly informative p~0.5', 3.0, 3.0),
    ('Wrong prior p~0.3', 5.0, 12.0),
]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (prior_name, a_p, b_p) in zip(axes, priors_bb):
    bbb = BayesianBetaBinomial(alpha_prior=a_p, beta_prior=b_p)
    bbb.update(coin_flips)
    theta_g = np.linspace(0.01, 0.99, N_GRID)
    prior_pdf = np.exp(beta_log_prob(theta_g, a_p, b_p))
    post_pdf = bbb.posterior_pdf(theta_g)
    ci_lo, ci_hi = bbb.credible_interval(0.95)
    post_mean = bbb.posterior_mean()
    post_map = bbb.posterior_map()
    ax.fill_between(theta_g, post_pdf,
                    where=(theta_g >= ci_lo) & (theta_g <= ci_hi),
                    alpha=0.25, color='blue', label='95% CI')
    ax.plot(theta_g, prior_pdf, 'g--', lw=1.5, alpha=0.7, label='Prior')
    ax.plot(theta_g, post_pdf, 'b-', lw=2, label='Posterior')
    ax.axvline(COIN_TRUE_P, color='r', ls=':', lw=2, label='True p')
    ax.axvline(post_mean, color='orange', ls='--', lw=1.5, label=f'Post mean={post_mean:.3f}')
    ax.axvline(post_map, color='purple', ls='-.', lw=1.5, label=f'MAP={post_map:.3f}')
    ax.axvline(n_heads_total / N_COIN_FLIPS, color='gray', ls=':', lw=1.5, label=f'MLE={n_heads_total/N_COIN_FLIPS:.3f}')
    ax.set_xlabel('theta')
    ax.set_ylabel('Density')
    ax.set_title(f'{prior_name}\n(n={N_COIN_FLIPS} flips)')
    ax.legend(fontsize=7, ncol=2)
    ax.grid(True, alpha=0.3)
    print(f'{prior_name}:')
    print(f'  Posterior: Beta({bbb.alpha_:.1f},{bbb.beta_:.1f})')
    print(f'  Mean={post_mean:.3f} | MAP={post_map:.3f} | 95%CI=[{ci_lo:.3f},{ci_hi:.3f}]')

plt.suptitle('Posterior with Different Priors (n=100 coin flips)', fontsize=12)
plt.tight_layout()
plt.savefig('beta_prior_sensitivity.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# ── Bayesian Linear Regression on noisy sin(x) data ──────────────────────────
print('=== Bayesian Linear Regression: sin(x) Recovery ===\n')

blr = BayesianLinearRegression(degree=POLY_DEGREE, alpha=ALPHA_PRIOR, beta_noise=BETA_NOISE)
blr.fit(X_linreg, y_linreg)
pred_mean, pred_std = blr.predict(X_test_linreg)

# MAP weights (ridge equivalent)
w_map = blr.map_weights(X_linreg, y_linreg)
Phi_test = build_polynomial_features(X_test_linreg, POLY_DEGREE)
y_map_pred = Phi_test @ w_map

# MLE (OLS, no regularisation)
Phi_train = build_polynomial_features(X_linreg, POLY_DEGREE)
w_mle: np.ndarray = np.linalg.lstsq(Phi_train, y_linreg, rcond=None)[0]
y_mle_pred: np.ndarray = Phi_test @ w_mle

log_lml = blr.log_marginal_likelihood(X_linreg, y_linreg)
print(f'Log marginal likelihood: {log_lml:.2f}')
print(f'MAP weights: {w_map.round(4)}')
print(f'Posterior mean should equal MAP: max diff = {np.max(np.abs(blr.m_N_ - w_map)):.2e}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.fill_between(X_test_linreg,
                pred_mean - 2 * pred_std,
                pred_mean + 2 * pred_std,
                alpha=0.2, color='blue', label='Bayesian +/-2 std')
ax.fill_between(X_test_linreg,
                pred_mean - pred_std,
                pred_mean + pred_std,
                alpha=0.35, color='blue')
ax.scatter(X_linreg, y_linreg, s=20, c='black', alpha=0.6, zorder=5, label='Training data')
ax.plot(X_test_linreg, y_true_linreg, 'r-', lw=2, label='True sin(x)')
ax.plot(X_test_linreg, pred_mean, 'b-', lw=2, label='Bayesian mean')
ax.plot(X_test_linreg, y_map_pred, 'g--', lw=1.5, label='MAP (ridge)')
ax.plot(X_test_linreg, y_mle_pred, 'm:', lw=1.5, label='MLE (OLS)')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title(f'Bayesian LinReg (degree={POLY_DEGREE}, n={N_LINREG})')
ax.legend(fontsize=8, ncol=2)
ax.set_ylim(-3, 3)
ax.grid(True, alpha=0.3)

# Epistemic vs aleatoric uncertainty decomposition
ax = axes[1]
aleatoric_std = np.full_like(X_test_linreg, 1.0 / np.sqrt(BETA_NOISE))
Phi_test_full = build_polynomial_features(X_test_linreg, POLY_DEGREE)
epistemic_var = np.sum(Phi_test_full @ blr.S_N_ * Phi_test_full, axis=1)
epistemic_std = np.sqrt(np.maximum(epistemic_var, 0.0))
ax.plot(X_test_linreg, pred_std, 'b-', lw=2, label='Total std')
ax.plot(X_test_linreg, aleatoric_std, 'r--', lw=1.5, label='Aleatoric (noise)')
ax.plot(X_test_linreg, epistemic_std, 'g-.', lw=1.5, label='Epistemic (weight unc.)')
# Mark training data locations
for xi in X_linreg:
    ax.axvline(xi, color='gray', alpha=0.1, lw=0.5)
ax.set_xlabel('x')
ax.set_ylabel('Standard deviation')
ax.set_title('Uncertainty Decomposition')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('bayesian_linreg.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# ── Posterior weight sampling: visualise uncertainty in regression functions ───
print('=== Sampling from BLR Posterior: Visualising Function Uncertainty ===\n')

N_WEIGHT_SAMPLES: int = 30   # Number of posterior weight samples to draw
rng_blr = np.random.RandomState(SEED + 7)

# Sample weight vectors from the posterior distribution N(m_N, S_N)
weight_samples: np.ndarray = rng_blr.multivariate_normal(blr.m_N_, blr.S_N_, size=N_WEIGHT_SAMPLES)
print(f'Sampled {N_WEIGHT_SAMPLES} weight vectors from posterior N(m_N, S_N).')
print(f'Weight samples shape: {weight_samples.shape}  (n_samples x n_weights)')

# Generate predictions for each sampled weight vector
Phi_test_samp = build_polynomial_features(X_test_linreg, POLY_DEGREE)
y_samples: np.ndarray = Phi_test_samp @ weight_samples.T  # (n_test, N_WEIGHT_SAMPLES)

# Also compare: prior samples (before seeing data) vs posterior samples
n_features_blr: int = POLY_DEGREE + 1
prior_cov_blr: np.ndarray = (1.0 / ALPHA_PRIOR) * np.eye(n_features_blr)
weight_samples_prior: np.ndarray = rng_blr.multivariate_normal(
    np.zeros(n_features_blr), prior_cov_blr, size=N_WEIGHT_SAMPLES,
)
y_samples_prior: np.ndarray = Phi_test_samp @ weight_samples_prior.T

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
for sj in range(N_WEIGHT_SAMPLES):
    ax.plot(X_test_linreg, y_samples_prior[:, sj], 'g-', alpha=0.2, lw=0.8)
ax.plot(X_test_linreg, y_true_linreg, 'r-', lw=2, label='True sin(x)', zorder=5)
ax.scatter(X_linreg, y_linreg, s=20, c='black', alpha=0.6, zorder=6)
ax.set_ylim(-6, 6)
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title(f'Prior Samples (alpha={ALPHA_PRIOR}): Before Seeing Data')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

ax = axes[1]
for sj in range(N_WEIGHT_SAMPLES):
    ax.plot(X_test_linreg, y_samples[:, sj], 'b-', alpha=0.2, lw=0.8)
ax.fill_between(X_test_linreg, pred_mean - 2 * pred_std, pred_mean + 2 * pred_std,
                alpha=0.15, color='blue')
ax.plot(X_test_linreg, pred_mean, 'b-', lw=2.5, label='Posterior mean', zorder=5)
ax.plot(X_test_linreg, y_true_linreg, 'r-', lw=2, label='True sin(x)', zorder=6)
ax.scatter(X_linreg, y_linreg, s=20, c='black', alpha=0.7, zorder=7)
ax.set_ylim(-3, 3)
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title(f'Posterior Samples (n={N_LINREG}): After Observing Data')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.suptitle('Bayesian LinReg — Prior vs Posterior Function Samples', fontsize=12)
plt.tight_layout()
plt.savefig('blr_prior_posterior_samples.png', dpi=100, bbox_inches='tight')
plt.show()

y_sample_mean: np.ndarray = y_samples.mean(axis=1)
y_sample_std: np.ndarray = y_samples.std(axis=1)
print(f'Sample mean vs analytical mean: max diff = {np.max(np.abs(y_sample_mean - pred_mean)):.4f}')
print(f'Sample std  vs analytical std : max diff = {np.max(np.abs(y_sample_std - pred_std)):.4f}')
print('Monte Carlo and analytic posterior predictive agree (up to sampling noise).')

---
## Part 4 — Evaluation & Analysis

## Part 4 — Evaluation & Analysis

We compare our from-scratch Bayesian linear regression against sklearn's `BayesianRidge`, analyze credible interval coverage, and study how the prior precision $\alpha$ affects the bias-variance tradeoff.

In [ ]:
# ── Sklearn BayesianRidge comparison ─────────────────────────────────────────
print('=== Bayesian LinReg: Our Implementation vs Sklearn ===\n')


# Our implementation
t0 = time.perf_counter()
blr_ours = BayesianLinearRegression(degree=POLY_DEGREE, alpha=ALPHA_PRIOR, beta_noise=BETA_NOISE)
blr_ours.fit(X_linreg, y_linreg)
pred_m_ours, pred_s_ours = blr_ours.predict(X_test_linreg)
time_ours = time.perf_counter() - t0

# Sklearn BayesianRidge (learns alpha and beta_noise from data)
t0 = time.perf_counter()
Phi_train_sk = PolynomialFeatures(degree=POLY_DEGREE, include_bias=True).fit_transform(
    X_linreg.reshape(-1, 1)
)
Phi_test_sk = PolynomialFeatures(degree=POLY_DEGREE, include_bias=True).fit_transform(
    X_test_linreg.reshape(-1, 1)
)
br_sk = BayesianRidge(max_iter=300)
br_sk.fit(Phi_train_sk, y_linreg)
pred_m_sk, pred_s_sk = br_sk.predict(Phi_test_sk, return_std=True)
time_sk = time.perf_counter() - t0

# Evaluation: RMSE on test grid against true sin(x)
rmse_ours = float(np.sqrt(np.mean((pred_m_ours - y_true_linreg) ** 2)))
rmse_sk = float(np.sqrt(np.mean((pred_m_sk - y_true_linreg) ** 2)))

print(f'  Our BLR  : RMSE={rmse_ours:.4f} | time={time_ours:.4f}s | alpha={ALPHA_PRIOR}')
print(f'  Sklearn  : RMSE={rmse_sk:.4f} | time={time_sk:.4f}s | alpha={br_sk.alpha_:.4f} (learned)')
print(f'  Note: sklearn learns alpha via evidence maximisation; we fix it.')

# Sklearn Ridge (MAP, deterministic)
lambda_map = ALPHA_PRIOR / BETA_NOISE
ridge_sk = Ridge(alpha=lambda_map)
ridge_sk.fit(Phi_train_sk, y_linreg)
pred_ridge = ridge_sk.predict(Phi_test_sk)
rmse_ridge = float(np.sqrt(np.mean((pred_ridge - y_true_linreg) ** 2)))
print(f'  Ridge MAP: RMSE={rmse_ridge:.4f} (no uncertainty)')

fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(X_linreg, y_linreg, s=20, c='black', alpha=0.7, label='Training data')
ax.plot(X_test_linreg, y_true_linreg, 'r-', lw=2, label='True sin(x)')
ax.fill_between(X_test_linreg, pred_m_ours - 2*pred_s_ours, pred_m_ours + 2*pred_s_ours,
                alpha=0.2, color='blue')
ax.plot(X_test_linreg, pred_m_ours, 'b-', lw=2, label=f'Our BLR (RMSE={rmse_ours:.3f})')
ax.fill_between(X_test_linreg, pred_m_sk - 2*pred_s_sk, pred_m_sk + 2*pred_s_sk,
                alpha=0.15, color='green')
ax.plot(X_test_linreg, pred_m_sk, 'g--', lw=2, label=f'Sklearn BayesianRidge (RMSE={rmse_sk:.3f})')
ax.plot(X_test_linreg, pred_ridge, 'm:', lw=2, label=f'Ridge MAP (RMSE={rmse_ridge:.3f})')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('Bayesian LinReg Comparison')
ax.legend(fontsize=9)
ax.set_ylim(-3, 3)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('blr_sklearn_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# ── Credible interval coverage analysis ───────────────────────────────────────
print('=== Credible Interval Coverage Analysis: Normal-Normal ===\n')

# Theoretical property: a 95% credible interval should contain the true value ~95% of the time
# when the model is correctly specified.
n_experiments: int = 1000
coverage_levels: list = [0.50, 0.80, 0.90, 0.95, 0.99]
prior_mu_0, prior_tau_sq = 0.0, 9.0  # Wide prior
data_std_sq = KNOWN_SIGMA ** 2

# For each sample size, estimate empirical coverage
sample_sizes_cov: list = [5, 10, 20, 50]
rng_cov = np.random.RandomState(SEED + 42)

coverage_results: list = []
for n_cov in sample_sizes_cov:
    for level in coverage_levels:
        inside_count: int = 0
        for _ in range(n_experiments):
            data_cov = rng_cov.normal(TRUE_MU, KNOWN_SIGMA, size=n_cov)
            bnn_cov = BayesianNormalNormal(prior_mu_0, prior_tau_sq, data_std_sq)
            bnn_cov.update(data_cov)
            lo_cov, hi_cov = bnn_cov.credible_interval(level)
            if lo_cov <= TRUE_MU <= hi_cov:
                inside_count += 1
        empirical_cov = inside_count / n_experiments
        coverage_results.append({'n': n_cov, 'level': level, 'empirical': empirical_cov})

df_cov = pd.DataFrame(coverage_results)
pivot_cov = df_cov.pivot(index='n', columns='level', values='empirical')

print('Empirical coverage (rows=n, cols=nominal level):')
print(pivot_cov.round(3).to_string())

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ax = axes[0]
for n_plot in sample_sizes_cov:
    df_n = df_cov[df_cov['n'] == n_plot]
    ax.plot(df_n['level'], df_n['empirical'], '-o', ms=6, label=f'n={n_plot}')
ax.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Perfect calibration')
ax.set_xlabel('Nominal coverage')
ax.set_ylabel('Empirical coverage')
ax.set_title('Credible Interval Coverage (Normal-Normal)')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

ax = axes[1]
im_cov = ax.imshow(pivot_cov.values, cmap='RdYlGn', vmin=0.8, vmax=1.0, aspect='auto')
ax.set_xticks(range(len(coverage_levels)))
ax.set_xticklabels([str(l) for l in coverage_levels])
ax.set_yticks(range(len(sample_sizes_cov)))
ax.set_yticklabels([str(n) for n in sample_sizes_cov])
for ri in range(len(sample_sizes_cov)):
    for ci in range(len(coverage_levels)):
        val = pivot_cov.values[ri, ci]
        ax.text(ci, ri, f'{val:.2f}', ha='center', va='center', fontsize=9)
plt.colorbar(im_cov, ax=ax, label='Empirical coverage')
ax.set_xlabel('Nominal coverage')
ax.set_ylabel('Sample size n')
ax.set_title('Coverage Heatmap')

plt.tight_layout()
plt.savefig('credible_interval_coverage.png', dpi=100, bbox_inches='tight')
plt.show()

print('\nWith correct model specification, Bayesian credible intervals are well-calibrated.')

In [ ]:
# ── Model comparison: BLR with different degrees via log marginal likelihood ───
print('=== Bayesian Model Comparison: Polynomial Degree ===\n')

degrees_test: list = [1, 2, 3, 4, 5, 6, 7, 8]
lml_scores: list = []
rmse_train: list = []
rmse_test_list: list = []

for deg in degrees_test:
    blr_deg = BayesianLinearRegression(degree=deg, alpha=ALPHA_PRIOR, beta_noise=BETA_NOISE)
    blr_deg.fit(X_linreg, y_linreg)
    lml = blr_deg.log_marginal_likelihood(X_linreg, y_linreg)
    pred_train, _ = blr_deg.predict(X_linreg)
    pred_test_deg, _ = blr_deg.predict(X_test_linreg)
    r_train = float(np.sqrt(np.mean((pred_train - y_linreg) ** 2)))
    r_test = float(np.sqrt(np.mean((pred_test_deg - y_true_linreg) ** 2)))
    lml_scores.append(lml)
    rmse_train.append(r_train)
    rmse_test_list.append(r_test)
    print(f'  degree={deg} | log_lml={lml:.1f} | RMSE_train={r_train:.4f} | RMSE_test={r_test:.4f}')

best_deg_lml = degrees_test[int(np.argmax(lml_scores))]
best_deg_test = degrees_test[int(np.argmin(rmse_test_list))]
print(f'\nBest degree by log LML: {best_deg_lml}')
print(f'Best degree by test RMSE: {best_deg_test}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.plot(degrees_test, lml_scores, 'b-o', ms=6)
ax.axvline(best_deg_lml, color='r', ls='--', label=f'Best LML degree={best_deg_lml}')
ax.set_xlabel('Polynomial degree')
ax.set_ylabel('Log marginal likelihood')
ax.set_title('Bayesian Model Selection via Log LML')
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(degrees_test, rmse_train, 'b-o', ms=6, label='Train RMSE')
ax.plot(degrees_test, rmse_test_list, 'r-s', ms=6, label='Test RMSE')
ax.axvline(best_deg_lml, color='gray', ls='--', label=f'LML peak deg={best_deg_lml}')
ax.set_xlabel('Polynomial degree')
ax.set_ylabel('RMSE')
ax.set_title('Train vs Test RMSE by Degree')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('blr_model_selection.png', dpi=100, bbox_inches='tight')
plt.show()

print('\nLog marginal likelihood naturally penalises overfitting (Occam factor).')
print('It balances data fit against model complexity without a separate test set.')

In [ ]:
# ── BLR Prior Precision Sensitivity: how alpha shapes the bias-variance tradeoff ──
print('=== Prior Precision Sensitivity: Effect of alpha on BLR Posterior ===\n')

alpha_sweep: list = [0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]
colors_alpha: list = ['#e41a1c', '#ff7f00', '#4daf4a', '#377eb8', '#984ea3', '#a65628']
labels_alpha: list = [f'alpha={a}' for a in alpha_sweep]

# Store results: mean predictions, std predictions, log marginal likelihood
pred_means_alpha: list = []
pred_stds_alpha: list = []
lml_alpha: list = []
weight_norms_alpha: list = []

for alpha_val in alpha_sweep:
    blr_a = BayesianLinearRegression(degree=POLY_DEGREE, alpha=alpha_val, beta_noise=BETA_NOISE)
    blr_a.fit(X_linreg, y_linreg)
    Phi_test_a = build_polynomial_features(X_test_linreg, POLY_DEGREE)
    mu_a, std_a = blr_a.predict(Phi_test_a)
    lml_val = blr_a.log_marginal_likelihood(
        build_polynomial_features(X_linreg, POLY_DEGREE), y_linreg,
    )
    pred_means_alpha.append(mu_a)
    pred_stds_alpha.append(std_a)
    lml_alpha.append(lml_val)
    weight_norms_alpha.append(float(np.linalg.norm(blr_a.m_N_)))

print('alpha      | log p(y|alpha) | ||m_N|| | mean std (test)')
print('-' * 60)
for a, lml_v, wn, std_v in zip(alpha_sweep, lml_alpha, weight_norms_alpha, pred_stds_alpha):
    mean_std = float(std_v.mean())
    print(f'alpha={a:<8.3g} | {lml_v:14.2f} | {wn:7.3f} | {mean_std:.4f}')

best_alpha_idx: int = int(np.argmax(lml_alpha))
print(f'\nBest alpha by log marginal likelihood: {alpha_sweep[best_alpha_idx]}')

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes_flat = axes.flatten()

for k, (alpha_val, mu_a, std_a, col, lbl) in enumerate(
    zip(alpha_sweep, pred_means_alpha, pred_stds_alpha, colors_alpha, labels_alpha)
):
    ax = axes_flat[k]
    ax.fill_between(
        X_test_linreg,
        mu_a - 2 * std_a,
        mu_a + 2 * std_a,
        alpha=0.20,
        color=col,
        label='95% CI',
    )
    ax.plot(X_test_linreg, mu_a, color=col, lw=2, label='Posterior mean')
    ax.plot(X_test_linreg, y_true_linreg, 'k--', lw=1.5, label='True sin(x)')
    ax.scatter(X_linreg, y_linreg, s=15, c='black', alpha=0.5, zorder=5)
    ax.set_ylim(-5, 5)
    ax.set_title(lbl, fontsize=10)
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

plt.suptitle('BLR Prior Precision Sensitivity (degree=4)', fontsize=12)
plt.tight_layout()
plt.savefig('blr_alpha_sensitivity.png', dpi=100, bbox_inches='tight')
plt.show()

# Log marginal likelihood vs alpha — this is the Bayesian way to select alpha automatically
fig2, ax2 = plt.subplots(figsize=(7, 4))
ax2.semilogx(alpha_sweep, lml_alpha, 'bo-', lw=2, ms=8)
ax2.axvline(x=alpha_sweep[best_alpha_idx], color='red', ls='--', lw=1.5,
            label=f'Best alpha={alpha_sweep[best_alpha_idx]}')
ax2.set_xlabel('Prior precision alpha (log scale)')
ax2.set_ylabel('Log marginal likelihood')
ax2.set_title('Bayesian Model Selection: Log p(y|alpha)')
ax2.legend()
ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('blr_lml_vs_alpha.png', dpi=100, bbox_inches='tight')
plt.show()
print('Low alpha = weak prior (high variance, overfitting risk).')
print('High alpha = strong prior toward zero (high bias, underfitting risk).')
print('Log marginal likelihood automatically penalises both extremes.')


In [ ]:
# ── Online vs Batch Bayesian updating: verifying sequential consistency ──────
print('=== Sequential vs Batch Bayesian Update Consistency ===\n')

# Key property: Bayesian updates are order-independent and batch-consistent.
# Updating on data[0..n-1] sequentially = updating on all n data points at once.
# We verify this for both Beta-Binomial and Normal-Normal.

# --- Beta-Binomial: sequential batch split ---
rng_seq = np.random.RandomState(SEED + 42)
n_total_seq: int = 80
true_p_seq: float = 0.70
data_seq: np.ndarray = rng_seq.binomial(1, true_p_seq, size=n_total_seq).astype(float)

batch_sizes: list = [5, 10, 20, 40]
n_batches_list: list = [n_total_seq // bs for bs in batch_sizes]

print('Beta-Binomial: batch-size independence check')
print(f'{"Batch size":<12} | {"Sequential alpha_n":<20} | {"Sequential beta_n":<20} | {"Match":<8}')
print('-' * 70)

alpha_0_seq, beta_0_seq = 2.0, 2.0   # prior
# Batch update: all 80 at once
heads_all: int = int(data_seq.sum())
tails_all: int = n_total_seq - heads_all
alpha_batch, beta_batch = beta_posterior(alpha_0_seq, beta_0_seq, heads_all, tails_all)

for bs in batch_sizes:
    a_curr, b_curr = alpha_0_seq, beta_0_seq
    for start in range(0, n_total_seq, bs):
        chunk = data_seq[start:start + bs]
        h_chunk = int(chunk.sum())
        t_chunk = len(chunk) - h_chunk
        a_curr, b_curr = beta_posterior(a_curr, b_curr, h_chunk, t_chunk)
    match_str = 'YES' if (abs(a_curr - alpha_batch) < 1e-10 and abs(b_curr - beta_batch) < 1e-10) else 'NO'
    print(f'{bs:<12} | {a_curr:<20.6f} | {b_curr:<20.6f} | {match_str:<8}')

print(f'{"BATCH (all)":<12} | {alpha_batch:<20.6f} | {beta_batch:<20.6f} |')

# --- Normal-Normal: sequential batch split ---
print()
print('Normal-Normal: batch-size independence check')
mu_0_seq, tau_0_sq_seq = 0.0, 10.0
true_mu_seq: float = 3.5
sigma_sq_seq: float = 2.0
n_nn_seq: int = 60
data_nn_seq: np.ndarray = rng_seq.normal(true_mu_seq, np.sqrt(sigma_sq_seq), size=n_nn_seq)

# Batch update: all at once
mu_n_batch, tau_n_sq_batch = normal_posterior_known_var(mu_0_seq, tau_0_sq_seq, sigma_sq_seq, data_nn_seq)

print(f'{"Batch size":<12} | {"Seq mu_n":<15} | {"Seq tau_n^2":<15} | {"Match":<8}')
print('-' * 55)
for bs in batch_sizes:
    mu_curr, tau_sq_curr = mu_0_seq, tau_0_sq_seq
    for start in range(0, n_nn_seq, bs):
        chunk = data_nn_seq[start:start + bs]
        mu_curr, tau_sq_curr = normal_posterior_known_var(mu_curr, tau_sq_curr, sigma_sq_seq, chunk)
    match_str = 'YES' if (abs(mu_curr - mu_n_batch) < 1e-9 and abs(tau_sq_curr - tau_n_sq_batch) < 1e-9) else 'NO'
    print(f'{bs:<12} | {mu_curr:<15.6f} | {tau_sq_curr:<15.8f} | {match_str:<8}')

print(f'{"BATCH (all)":<12} | {mu_n_batch:<15.6f} | {tau_n_sq_batch:<15.8f} |')

# Visualise convergence of the sequential posterior mean vs n
alphas_track: list = [alpha_0_seq]
betas_track: list = [beta_0_seq]
posterior_means_track: list = [alpha_0_seq / (alpha_0_seq + beta_0_seq)]
for flip in data_seq:
    a_t, b_t = beta_posterior(alphas_track[-1], betas_track[-1],
                               int(flip), int(1 - flip))
    alphas_track.append(a_t)
    betas_track.append(b_t)
    posterior_means_track.append(a_t / (a_t + b_t))

ci_lo_track: list = []
ci_hi_track: list = []
for a_t, b_t in zip(alphas_track, betas_track):
    lo, hi = stats.beta.ppf([0.025, 0.975], a_t, b_t)
    ci_lo_track.append(lo)
    ci_hi_track.append(hi)

ci_lo_arr = np.array(ci_lo_track)
ci_hi_arr = np.array(ci_hi_track)
pm_arr = np.array(posterior_means_track)
n_range = np.arange(len(pm_arr))

fig, ax = plt.subplots(figsize=(9, 4))
ax.fill_between(n_range, ci_lo_arr, ci_hi_arr, alpha=0.25, color='steelblue', label='95% credible interval')
ax.plot(n_range, pm_arr, 'b-', lw=2, label='Posterior mean')
ax.axhline(y=true_p_seq, color='red', ls='--', lw=2, label=f'True p = {true_p_seq}')
ax.set_xlabel('Number of coin flips observed')
ax.set_ylabel('Estimated P(heads)')
ax.set_title('Sequential Bayesian Updating: Beta-Binomial Convergence')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('sequential_bb_convergence.png', dpi=100, bbox_inches='tight')
plt.show()

ci_width_final: float = float(ci_hi_arr[-1] - ci_lo_arr[-1])
ci_width_half: float = float(ci_hi_arr[n_total_seq // 2] - ci_lo_arr[n_total_seq // 2])
print(f'CI width at n=40: {ci_width_half:.4f}')
print(f'CI width at n=80: {ci_width_final:.4f}')
print(f'Width halved? {ci_width_half / ci_width_final:.2f}x wider at half the data.')
print('Sequential updates are exactly equivalent to batch updates for conjugate priors.')


---
## Part 5 — Summary & Lessons Learned

### Key Takeaways

1. **Bayes' theorem unifies prior knowledge and data.** The posterior is proportional to likelihood × prior; with conjugate priors, this update is analytic and exact. The posterior mean is always a weighted average of the prior mean and the MLE, with weights set by relative precisions.

2. **MAP estimation is penalised MLE.** The MAP estimate maximises the posterior but discards all uncertainty — it is equivalent to $L_2$-regularised maximum likelihood (ridge regression for linear models). With enough data, MLE, MAP, and the full posterior all converge.

3. **Full posteriors quantify uncertainty.** Credible intervals from the posterior have a direct probability interpretation: "the true parameter lies in this interval with 95% posterior probability." With correct model specification, they are well-calibrated empirically.

4. **Bayesian linear regression decomposes uncertainty.** The predictive variance has two parts: aleatoric uncertainty (irreducible observation noise) and epistemic uncertainty (uncertainty about the weights). Epistemic uncertainty is high in data-sparse regions and shrinks with more data.

5. **Log marginal likelihood enables model selection.** By integrating out the weights, the evidence penalises overfitting through an "Occam factor" — automatically balancing data fit against model complexity without a held-out validation set.

### What's Next

→ **Module 04 (ML Theory & Evaluation)** builds on these probabilistic foundations — bias-variance tradeoff, learning curves, and cross-validation all have Bayesian interpretations developed here.

→ **Module 11 (Variational Autoencoders)** extends Bayesian inference to neural networks: the ELBO objective is exactly a Bayesian evidence lower bound, and the encoder network approximates the posterior $p(z | x)$.

### Going Further

- Bishop (2006): *Pattern Recognition and Machine Learning*, Chapter 3 — full Bayesian linear regression derivation.
- Gelman et al. (2013): *Bayesian Data Analysis* — the standard reference for applied Bayesian inference.
- MacKay (1992): *Bayesian interpolation* — evidence framework and hyperparameter optimisation via type-II MLE.